# Feature Engineering Notebook
Reusable transformations for League of Legends early-game stats.

## Engineering goals
- Encode *relative advantage* (shares, ratios, differences) to highlight which team owns tempo(ahead at the moment kinda).
- Capture *interaction proxies* like objective control weighted by gold diff.(whats most important obj wise)
- Provide a single function (`engineer_features`) every other notebook can reuse by copy/paste.

### Feature checklist
1. Objective share (`blue_objective_share`, `red_objective_share`)
2. Vision share & ward efficiency
3. Kill participation proxies
4. Economy rates (gold per kill, XP per minute)
5. Momentum interactions (gold diff × objectives, XP diff × kills)
6. Normalized versions (per-minute, per-death) for stability


In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Could not find project root containing data/')
    REPO_ROOT = REPO_ROOT.parent

print(f'Using repo root: {REPO_ROOT}')
DATA_PATH = REPO_ROOT / 'data/raw/high_diamond_ranked_10min.csv'
BASE_DF = pd.read_csv(DATA_PATH)
TARGET = 'blueWins'
BASE_DF.head()


Using repo root: /Users/liamsandy/ML_Project


,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,...,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,...,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,...,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,...,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,...,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4


## Helper: safe division
Avoid divide-by-zero when creating ratios.

In [5]:
def safe_ratio(numerator, denominator, fill_value=0.0):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).fillna(fill_value)


## Main feature engineering function
Copy this function into other notebooks

In [8]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    features = df.copy()

    # Objective share: proportion of total neutral objectives taken by blue/red
    blue_objectives = features['blueDragons'] + features['blueHeralds'] + features['blueEliteMonsters']
    red_objectives = features['redDragons'] + features['redHeralds'] + features['redEliteMonsters']
    total_objectives = blue_objectives + red_objectives
    features['blue_objective_share'] = safe_ratio(blue_objectives, total_objectives, 0.5)
    features['red_objective_share'] = safe_ratio(red_objectives, total_objectives, 0.5)

    # Vision share & efficiency
    total_wards = features['blueWardsPlaced'] + features['redWardsPlaced']
    features['blue_vision_share'] = safe_ratio(features['blueWardsPlaced'], total_wards, 0.5)
    features['red_vision_share'] = safe_ratio(features['redWardsPlaced'], total_wards, 0.5)
    features['blue_ward_efficiency'] = safe_ratio(features['blueWardsDestroyed'], features['blueWardsPlaced'], 0.0)
    features['red_ward_efficiency'] = safe_ratio(features['redWardsDestroyed'], features['redWardsPlaced'], 0.0)

    # Kill participation proxies
    total_team_kills = features['blueKills'] + features['redKills']
    features['blue_kill_share'] = safe_ratio(features['blueKills'], total_team_kills, 0.5)
    features['red_kill_share'] = safe_ratio(features['redKills'], total_team_kills, 0.5)

    # Economy ratios
    features['blue_gold_per_kill'] = safe_ratio(features['blueTotalGold'], features['blueKills'] + 1)
    features['red_gold_per_kill'] = safe_ratio(features['redTotalGold'], features['redKills'] + 1)
    features['blue_xp_per_min'] = features['blueTotalExperience'] / 10.0
    features['red_xp_per_min'] = features['redTotalExperience'] / 10.0

    # Momentum interactions
    features['gold_obj_momentum'] = (features['blueGoldDiff']) * (features['blueDragons'] + features['blueHeralds'])
    features['xp_kill_momentum'] = features['blueExperienceDiff'] * features['blueKills']

    # Normalized diffs (per minute)
    features['gold_diff_per_min'] = features['blueGoldDiff'] / 10.0
    features['xp_diff_per_min'] = features['blueExperienceDiff'] / 10.0

    # Drop raw identifier if present
    if 'gameId' in features:
        features = features.drop(columns=['gameId'])

    return features


## Testing
The resulting DataFrame is what downstream notebooks should use.

In [13]:
ENGINEERED = engineer_features(BASE_DF)
ENGINEERED.describe().T.head(10)


,count,mean,std,min,25%,50%,75%,max
blueWins,9879.0,0.499038,0.500024,0.0,0.0,0.0,1.0,1.0
blueWardsPlaced,9879.0,22.288288,18.019177,5.0,14.0,16.0,20.0,250.0
blueWardsDestroyed,9879.0,2.824881,2.174998,0.0,1.0,3.0,4.0,27.0
blueFirstBlood,9879.0,0.504808,0.500002,0.0,0.0,1.0,1.0,1.0
blueKills,9879.0,6.183925,3.011028,0.0,4.0,6.0,8.0,22.0
blueDeaths,9879.0,6.137666,2.933818,0.0,4.0,6.0,8.0,22.0
blueAssists,9879.0,6.645106,4.064520,0.0,4.0,6.0,9.0,29.0
blueEliteMonsters,9879.0,0.549954,0.625527,0.0,0.0,0.0,1.0,2.0
blueDragons,9879.0,0.361980,0.480597,0.0,0.0,0.0,1.0,1.0
blueHeralds,9879.0,0.187974,0.390712,0.0,0.0,0.0,0.0,1.0


## Save for reuse
Stores both full dataset and training/test splits so other notebooks can load without re-running this notebook.

In [15]:
output_dir = REPO_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
full_path = output_dir / 'engineered_features.csv'
ENGINEERED.to_csv(full_path, index=False)
print(f'Saved engineered features to {full_path}')

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    ENGINEERED,
    test_size=0.2,
    stratify=ENGINEERED[TARGET],
    random_state=42,
)
train_df.to_csv(output_dir / 'engineered_train.csv', index=False)
test_df.to_csv(output_dir / 'engineered_test.csv', index=False)
print('Train/Test CSVs updated.')


Saved engineered features to /Users/liamsandy/ML_Project/data/processed/engineered_features.csv
Train/Test CSVs updated.


### Reusing in other notebooks
```python
from pathlib import Path
import pandas as pd

data_path = Path('../data/processed/engineered_features.csv')  # adjust relative path
features = pd.read_csv(data_path)
```
Or copy the `engineer_features` function directly if you need to recompute on-the-fly.